# Download data to /data, extract them

In [5]:
import urllib.request
import zipfile
from pathlib import Path

In [8]:
HOME_DATA_DIR = Path.cwd().parent.parent.resolve()
DATA_DIR = Path(HOME_DATA_DIR / "data")
ZIP_DIR = DATA_DIR / "zips"
EXTRACT_DIR = DATA_DIR / "extracts"

DATASETS = {
    # --- Fluorescence (2D + time) ---
    "Fluo-C2DL-Huh7_train": "https://data.celltrackingchallenge.net/training-datasets/Fluo-C2DL-Huh7.zip",
    "Fluo-C2DL-Huh7_test": "https://data.celltrackingchallenge.net/test-datasets/Fluo-C2DL-Huh7.zip",
    "Fluo-C2DL-MSC_train": "https://data.celltrackingchallenge.net/training-datasets/Fluo-C2DL-MSC.zip",
    "Fluo-C2DL-MSC_test": "https://data.celltrackingchallenge.net/test-datasets/Fluo-C2DL-MSC.zip",
    "Fluo-N2DH-GOWT1_train": "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip",
    "Fluo-N2DH-GOWT1_test": "https://data.celltrackingchallenge.net/test-datasets/Fluo-N2DH-GOWT1.zip",
    "Fluo-N2DL-HeLa_train": "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip",
    "Fluo-N2DL-HeLa_test": "https://data.celltrackingchallenge.net/test-datasets/Fluo-N2DL-HeLa.zip",
}


def ensure_dirs() -> None:
    ZIP_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def download_if_missing(name: str, url: str) -> Path:
    zip_path = ZIP_DIR / f"{name}.zip"
    if zip_path.exists():
        print(f"Zip exists, skipping download: {zip_path.name}")
        return zip_path

    print(f"Downloading {name}...")
    urllib.request.urlretrieve(url, zip_path)
    return zip_path


def extract_if_missing(name: str, zip_path: Path) -> Path:
    out_dir = EXTRACT_DIR / name
    if out_dir.exists():
        print(f"Extract exists, skipping unzip: {out_dir.name}")
        return out_dir

    print(f"Extracting {zip_path.name}...")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(out_dir)
    return out_dir


def main() -> None:
    ensure_dirs()

    for dataset_name, dataset_url in DATASETS.items():
        zip_path = download_if_missing(dataset_name, dataset_url)
        extract_if_missing(dataset_name, zip_path)

    print(f"Zip folder: {ZIP_DIR}")
    print(f"Extract folder: {EXTRACT_DIR}")


if __name__ == "__main__":
    main()

Extracting Fluo-C2DL-Huh7_train.zip...
Extracting Fluo-C2DL-Huh7_test.zip...
Extracting Fluo-C2DL-MSC_train.zip...
Extracting Fluo-C2DL-MSC_test.zip...
Extracting Fluo-N2DH-GOWT1_train.zip...
Extracting Fluo-N2DH-GOWT1_test.zip...
Extracting Fluo-N2DL-HeLa_train.zip...
Extracting Fluo-N2DL-HeLa_test.zip...
Zip folder: E:\cv\data\zips
Extract folder: E:\cv\data\extracts


# We are training a segmentation pipeline, with the concepts of:
- Automatic thresholding of picture, to improve the performance of segmentation
- CNN to detect cell content areas.
# We'll train the CNN first.

# Data preparation steps:
- Download data
- Unzip
- Move ground truth to one folder

In [14]:
paths = [
	[r"data\extracts\Fluo-C2DL-Huh7_train\Fluo-C2DL-Huh7\01",	r"data\extracts\Fluo-C2DL-Huh7_train\Fluo-C2DL-Huh7\01_GT\SEG"],
	[r"data\extracts\Fluo-C2DL-Huh7_train\Fluo-C2DL-Huh7\02",	r"data\extracts\Fluo-C2DL-Huh7_train\Fluo-C2DL-Huh7\02_GT\SEG"],
	[r"data\extracts\Fluo-C2DL-MSC_train\Fluo-C2DL-MSC\01",		r"data\extracts\Fluo-C2DL-MSC_train\Fluo-C2DL-MSC\01_GT\SEG"],
	[r"data\extracts\Fluo-C2DL-MSC_train\Fluo-C2DL-MSC\02",		r"data\extracts\Fluo-C2DL-MSC_train\Fluo-C2DL-MSC\02_GT\SEG"],
	[r"data\extracts\Fluo-N2DH-GOWT1_train\Fluo-N2DH-GOWT1\01",	r"data\extracts\Fluo-N2DH-GOWT1_train\Fluo-N2DH-GOWT1\01_GT\SEG"],
	[r"data\extracts\Fluo-N2DH-GOWT1_train\Fluo-N2DH-GOWT1\02",	r"data\extracts\Fluo-N2DH-GOWT1_train\Fluo-N2DH-GOWT1\02_GT\SEG"],
	[r"data\extracts\Fluo-N2DL-HeLa_train\Fluo-N2DL-HeLa\01",	r"data\extracts\Fluo-N2DL-HeLa_train\Fluo-N2DL-HeLa\01_GT\SEG"],
	[r"data\extracts\Fluo-N2DL-HeLa_train\Fluo-N2DL-HeLa\02",	r"data\extracts\Fluo-N2DL-HeLa_train\Fluo-N2DL-HeLa\02_GT\SEG"]
]

In [17]:
import re
import shutil
from pathlib import Path

# Output folders
OUT_IMGS = DATA_DIR / "imgs"
OUT_MASKS = DATA_DIR / "masks"
OUT_IMGS.mkdir(parents=True, exist_ok=True)
OUT_MASKS.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}
ID_RE = re.compile(r"(\d{3})$")


def frame_id(path_obj: Path):
    m = ID_RE.search(path_obj.stem)
    return m.group(1) if m else None


def resolve_input_path(raw_path: str) -> Path:
    p = Path(raw_path)
    if p.is_absolute():
        return p

    # Treat relative "data/..." paths as project-root relative (same root as DATA_DIR).
    parts_lower = [part.lower() for part in p.parts]
    if parts_lower and parts_lower[0] == "data":
        return (DATA_DIR.parent / p).resolve()

    # Fallback: notebook current working directory relative.
    return (Path.cwd() / p).resolve()


def collect_by_id(folder: Path):
    mapping = {}
    for p in folder.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        fid = frame_id(p)
        if fid is None:
            continue
        mapping[fid] = p
    return mapping


total_copied = 0
for img_dir_str, seg_dir_str in paths:
    img_dir = resolve_input_path(img_dir_str)
    seg_dir = resolve_input_path(seg_dir_str)

    if not img_dir.exists() or not seg_dir.exists():
        print(f"Skipping missing pair: {img_dir} | {seg_dir}")
        continue

    dataset_name = img_dir.parent.name
    imgs_by_id = collect_by_id(img_dir)
    segs_by_id = collect_by_id(seg_dir)
    matched_ids = sorted(set(imgs_by_id.keys()) & set(segs_by_id.keys()))

    copied_this_pair = 0
    for fid in matched_ids:
        img_src = imgs_by_id[fid]
        mask_src = segs_by_id[fid]
        out_name = f"{dataset_name}_{fid}{img_src.suffix.lower()}"
        out_mask_name = f"{dataset_name}_{fid}{mask_src.suffix.lower()}"

        shutil.copy2(img_src, OUT_IMGS / out_name)
        shutil.copy2(mask_src, OUT_MASKS / out_mask_name)
        copied_this_pair += 1

    total_copied += copied_this_pair
    print(
        f"{dataset_name}: matched {copied_this_pair} | "
        f"imgs={len(imgs_by_id)} segs={len(segs_by_id)} "
        f"unmatched_imgs={len(set(imgs_by_id) - set(segs_by_id))} "
        f"unmatched_segs={len(set(segs_by_id) - set(imgs_by_id))}"
    )

print(f"Total matched copies: {total_copied}")
print(f"Images folder: {OUT_IMGS.resolve()}")
print(f"Masks folder: {OUT_MASKS.resolve()}")

Fluo-C2DL-Huh7: matched 8 | imgs=30 segs=8 unmatched_imgs=22 unmatched_segs=0
Fluo-C2DL-Huh7: matched 5 | imgs=30 segs=5 unmatched_imgs=25 unmatched_segs=0
Fluo-C2DL-MSC: matched 18 | imgs=48 segs=18 unmatched_imgs=30 unmatched_segs=0
Fluo-C2DL-MSC: matched 33 | imgs=48 segs=33 unmatched_imgs=15 unmatched_segs=0
Fluo-N2DH-GOWT1: matched 30 | imgs=92 segs=30 unmatched_imgs=62 unmatched_segs=0
Fluo-N2DH-GOWT1: matched 20 | imgs=92 segs=20 unmatched_imgs=72 unmatched_segs=0
Fluo-N2DL-HeLa: matched 28 | imgs=92 segs=28 unmatched_imgs=64 unmatched_segs=0
Fluo-N2DL-HeLa: matched 8 | imgs=92 segs=8 unmatched_imgs=84 unmatched_segs=0
Total matched copies: 150
Images folder: E:\cv\data\imgs
Masks folder: E:\cv\data\masks
